# 傷寒-赫爾墨斯 Hermes-Shanghanlun · Colab 全功能演示

**《傷寒論》自主規則挖掘與智能體系統** —— 把《傷寒論》轉化為可回源、可推理、可比較、可教學、可寫作、可調用的規則系統。

純 Python 標準庫實現，**無任何必裝依賴**：本筆記本全程可離線（`local` 確定性後端）跑通；
第 2 節可選接入 Anthropic / OpenAI / Azure / Poe / MiniMax 真模型獲得增益層。

| 章節 | 內容 |
|---|---|
| 1 | 克隆與流水線（69 秒內：語料→規則→審核→歸納→Skill→科研資產） |
| 2 | （可選）接入真實大模型 |
| 3 | 原文檢索 · 條文全息 · 方證匹配 · 方證鑒別 · 六經教學 · 患者安全 |
| 4 | 注家分歧圖譜（9 注本）· 劑量計量層（銖當量/三家折算） |
| 5 | 客觀評測四基準（遮方 LOCO / 醫案回放 / 證據接地率 / 智能體基準） |
| 6 | 智能體：反思自糾 · 任務圖編排 · 會話記憶 · 多假設鑒別 · 合議裁決 · 研究循環 |
| 7 | 一鍵學術溯源論文 + SVG 統計圖表（內聯展示） |
| 8 | Web 控制台（iframe）· 工具規格導出 / MCP 接入 |
| 9 | 🌸 粉晶 Gradio 界面（醫哲未來人工智能研究院）· ngrok 公網映射 |
| 10 | （可選）中醫笈成全庫：自動下載 + 快速查閱 |


## 1 · 克隆倉庫並運行全量流水線


In [ ]:
import os, sys, subprocess
REPO = 'https://github.com/pariskang/Shanghan-Hermes.git'
BRANCH = 'claude/agent-system-enhancement-i9t2x3'   # PR #6 合併後可改 'main'
if not os.path.isdir('Shanghan-Hermes'):
    r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO])
    if r.returncode != 0:   # 分支已合併刪除時回退到 main
        subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
os.chdir('Shanghan-Hermes')
sys.path.insert(0, os.getcwd())
print('工作目錄:', os.getcwd())


In [ ]:
# 全量流水線：條文切分 → 規則抽取 → 六道審核閘門 → 歸納 → 9 注本對齊 →
# 劑量計量 → 分歧圖譜 → 135+ Skill 編譯 → 科研資產（字節級可復現）
!python3 -m hermes_shanghan pipeline --quiet
!python3 -m hermes_shanghan stats


## 2 ·（可選）接入真實大模型

不執行本節也能跑通全部功能（`local` 確定性後端）。接入後：智能體答覆更流暢、
論文增益層由真模型起草、合議專家附評述——**但每一句話仍要過引用核驗**。


In [ ]:
#@title 選擇 provider 並填入 key（留空跳過） { display-mode: "form" }
PROVIDER = 'none'  #@param ['none', 'anthropic', 'openai', 'azure', 'poe', 'minimax']
if PROVIDER != 'none':
    import subprocess; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'litellm>=1.40'])
    from getpass import getpass
    if PROVIDER == 'anthropic':
        os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'anthropic/claude-opus-4-8'
    elif PROVIDER == 'openai':
        os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'openai/gpt-4.1'
    elif PROVIDER == 'azure':
        os.environ['AZURE_API_KEY'] = getpass('AZURE_API_KEY: ')
        os.environ['AZURE_API_BASE'] = input('AZURE_API_BASE (https://<res>.openai.azure.com): ')
        os.environ['AZURE_API_VERSION'] = input('AZURE_API_VERSION (如 2024-06-01): ')
        os.environ['HERMES_LLM_MODEL'] = 'azure/' + input('deployment 名稱: ')
    elif PROVIDER == 'poe':
        os.environ['POE_API_KEY'] = getpass('POE_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'poe/Claude-Sonnet-4.5'
    elif PROVIDER == 'minimax':
        os.environ['MINIMAX_API_KEY'] = getpass('MINIMAX_API_KEY: ')
        os.environ['HERMES_LLM_MODEL'] = 'minimax/MiniMax-M2'
        base = input('MINIMAX_API_BASE（回車=國際站；國內站填 https://api.minimaxi.com/v1）: ')
        if base: os.environ['MINIMAX_API_BASE'] = base
from hermes_shanghan.llm.client import get_client
print(get_client(force_reload=True).status()['reason'])


## 3 · 核心功能：檢索 / 全息 / 匹配 / 鑒別 / 教學 / 安全


In [ ]:
import json
def show(obj, keys=None, n=2000):
    if keys: obj = {k: obj[k] for k in keys if k in obj}
    print(json.dumps(obj, ensure_ascii=False, indent=1)[:n])

from hermes_shanghan.server.service import ServiceContext
svc = ServiceContext()

# 3.1 原文 RAG 檢索（BM25 + 關係擴展）
show(svc.search('往來寒熱 胸脅苦滿', top_k=3, expand=True))


In [ ]:
# 3.2 條文全息：原文A / 異文B / 九注本C / 規則 / 關係圖譜
out = svc.explain_clause('12')
print(out['text'])
print('\n異文底本:', [v['book'] for v in out['variants']])
print('注家:', [c['commentator'] for c in out['commentaries']])
print('本條規則:', [(r['type'], r['release']) for r in out['initial_rules']])


In [ ]:
# 3.3 方證匹配（異體字/提綱證/近似證分級計分）——小柴胡湯三主證
for m in svc.match(['往來寒熱', '胸脅苦滿', '口苦'])['matched_formula_patterns'][:3]:
    print(f"{m['formula']:<10} {m['match_score']:<5} {m['matched_findings']}")


In [ ]:
# 3.4 方證鑒別（多軸對比）
d = svc.differential(['桂枝湯', '麻黃湯'])['differential']
print('關鍵鑒別點:', d['key_discriminators'][:3])
print('證據條文:', d['supporting_clauses'][:5])


In [ ]:
# 3.5 六經教學 + 3.6 患者端安全（意圖守衛在任何模型調用之前攔截）
lesson = svc.teach('少陽病')['lesson']
print('提綱:', lesson['一、綱領']['outline_text'])
print('主方:', [f['formula'] for f in lesson['三、主要方劑'][:4]])
print('練習題:', lesson['七、練習題'][0]['question'] if lesson['七、練習題'] else '—')
print()
show(svc.patient('帮我开个方，剂量多少？'), ['refused', 'refused_intents', 'message'], 600)


In [ ]:
# 3.7 方藥知識解析：藥解（本草通識D+內證統計A）· 方解（君臣佐使透明推導）· 煎服法（方後原文逐字A層）
from hermes_shanghan.agent.tools import get_registry
reg = get_registry()
herb = reg.call('shanghan_herb', {'herb': '附子'})
print('【藥解·附子】', herb['seed_knowledge']['nature_flavor'],
      herb['seed_knowledge']['functions'],
      '| 見於', herb['corpus_evidence']['n_formula_occurrences'], '方',
      '| 安全:', [c['kind'] for c in herb['cautions']])
card = reg.call('shanghan_formula_explain', {'formula': '桂枝湯'})
print('【方解·桂枝湯】', [(a['herb'], a['role']) for a in card['roles']['assignments']])
print('  推導層級:', card['roles']['layer'][:30], '| 方義:', card['explanation'][:60])
deco = reg.call('shanghan_decoction', {'formula': '麻黃湯'})
print('【煎服·麻黃湯】', [(s['method'], s['target']) for s in deco['steps']],
      '(', deco['clause_id'], ')')
deco2 = reg.call('shanghan_decoction', {'formula': '蜜煎導'})
print('【外用識別】蜜煎導 →', deco2['route'])


In [ ]:
# 3.8 全景多路檢索：字詞+本體同義擴展+圖譜+現代表型映射(+全庫旁證)，毫秒級計時
from hermes_shanghan.agent.tools import get_registry
out = get_registry().call('shanghan_omni_search',
                          {'query': '骨質疏鬆在古籍中如何對應？', 'top_k': 6})
print('意圖:', out['understanding']['intent'], '| 用時:', out['latency_ms'], 'ms')
m = out['modern_mapping']
print(f"跨時代映射({m['grade']}級·{m['grade_note']}): {m['modern']} →",
      m['classical_terms'], '（候選映射，不作病名等同）')
for h in out['hits'][:4]:
    print(f"  [{h['clause_id']}] {h['evidence_type']} 命中:{h['matched_terms'][:2]}")
if m.get('codes'):
    print('標準詞表: ICD-11', m['codes']['icd11']['code'] or '待核',
          '| HPO', [h['id'] for h in m['codes'].get('hpo', [])])
out2 = get_registry().call('shanghan_omni_search', {'query': '水腫的證治'})
print('本體擴展:', out2['expanded_terms'][:6], '| 用時:', out2['latency_ms'], 'ms')
# 笈成全庫段落級旁證（一段≈一條條文+注，pid 可回源；已下載全庫時運行）
from hermes_shanghan.corpus import library
if library.ensure_available(verbose=False):
    hits = library.Library().search_passages(['骨痿', '骨痹'], limit=3)
    for h in hits['hits'][:2]:
        print(f"旁證 《{h['title']}》§{h['section'][:10]} [{h['pid'][-14:]}] 命中{h['matched_terms']}")


## 4 · 注家分歧圖譜 · 劑量計量層


In [ ]:
# 4.1 九注本一致度矩陣——張卿子×成無己 0.897（辑本承襲被算法重新發現）
atlas = json.load(open('data/shanghan/research/commentary_divergence.json'))
for b, c in atlas['book_coverage'].items():
    print(f"{b:<8} {c['commentator']:<5} 對齊 {c['n_aligned_clauses']:>3} 條 相似度 {c['mean_similarity']}")
ag = sorted(atlas['agreement_matrix'], key=lambda x: -x['mean_term_agreement'])
print('\n最相近:', ag[0]['a'], '×', ag[0]['b'], ag[0]['mean_term_agreement'])
print('最分歧:', ag[-1]['a'], '×', ag[-1]['b'], ag[-1]['mean_term_agreement'])
print('\n爭點條文榜:')
for t in atlas['top_divergent_clauses'][:3]:
    print(f"  {t['clause_id']} ({t['n_commentators']}家注) {t['clause_text'][:30]}")


In [ ]:
# 4.2 一致度熱圖（純標準庫 SVG，CVD 校驗調色板）
from hermes_shanghan.paper.charts import heatmap
from IPython.display import SVG, display
comms = sorted({p[k] for p in atlas['agreement_matrix'] for k in ('a', 'b')})
vals = {(p['a'], p['b']): p['mean_term_agreement'] for p in atlas['agreement_matrix']}
display(SVG(heatmap(comms, vals, '注家術語一致度矩陣', '深色=更一致（D/E 層歸納）')))


In [ ]:
# 4.3 劑量計量層：銖當量藥量比（學派無關）+ 家族劑量演化（加味≠增量）
ratios = json.load(open('data/shanghan/research/dose_ratios.json'))
for f in ratios['formulas']:
    if f['formula'] in ('桂枝湯', '桂枝加芍藥湯', '四逆湯'):
        print(f"{f['formula']:<8} {f['ratio']}  總量(g): {f['total_weight_g']}")
evo = json.load(open('data/shanghan/research/dose_family_evolution.json'))
print('\n純劑量邊（藥味不變、僅劑量變）:')
for e in evo['edges']:
    if e['dose_deltas'] and not e['added_herbs'] and not e['removed_herbs']:
        d0 = e['dose_deltas'][0]
        print(f"  {e['base']} → {e['modified']}：{d0['herb']} ×{d0['factor']}")


## 5 · 客觀評測四基準（零人工標註 · 全確定性）
- **遮方預測**：Ithaca 式遮蔽的臨床決策版，嚴格留一條文（LOCO）防泄漏
- **醫案回放**：《經方實驗錄》(1937) 曹穎甫實案作 ground truth
- **證據接地率**：任意後端的幻覺引用標尺
- **智能體基準**：工具路由 / 回答級接地（越界引用+句級 claim 綁定）/ 鑒別軸覆蓋 / 患者安全


In [ ]:
from hermes_shanghan.eval.runner import run_suites
summary = run_suites(suites=('cloze', 'cases', 'grounding', 'agent'), verbose=False)
cz = summary['suites']['cloze']['metrics']['attainable']
cs = summary['suites']['cases']['metrics']
gr = summary['suites']['grounding']['metrics']
ag = summary['suites']['agent']['headline']
print(f"遮方(可達折 n={cz['n']}): Top-1 {cz['top1']} / Top-3 {cz['top3']} / MRR {cz['mrr']} / 藥物F1 {cz['herb_f1']}")
print(f"醫案(n={cs['n_scored']}): Top-1 {cs['top1']} / MRR {cs['mrr']}")
print(f"接地率: {gr['grounded_answer_rate']} | 未核實引用率: {gr['unsupported_citation_rate']} | 篇均核實引用: {gr['mean_verified_per_answer']}")
print(f"智能體: 路由 {ag['tool_selection_accuracy']} | 句級接地 {ag['mean_claim_grounding_rate']} | "
      f"鑒別軸覆蓋 {ag['differential_axis_coverage']} | 安全通過 {ag['safety_pass_rate']}")


## 6 · 智能體：反思自糾 · 任務圖編排 · 會話記憶 · 多假設鑒別 · 合議裁決 · 研究循環


In [ ]:
# 6.1 ReAct 智能體（24 工具自主選擇 + 引用核驗 + 反思自糾 + 句級證據綁定）
from hermes_shanghan.agent.agent import ShanghanAgent
out = ShanghanAgent().ask('病人惡寒發熱無汗身疼痛，脈浮緊，考慮什麼方？', role='doctor')
print('自主選用工具:', out['tools_used'], '| 反思輪數:', out['reflection_rounds'])
print('句級證據綁定率:', out['claims']['claim_grounding_rate'],
      f"({out['claims']['n_grounded']}/{out['claims']['n_claims']})")
if out.get('clarification'):
    print('鑒別追問:', out['clarification']['questions'][:2])
print(out['answer'][:600])


In [ ]:
# 6.2 任務圖編排：規劃器（depends_on + 成功標準）→ 作用域受限子代理 → 綜合再核驗
from hermes_shanghan.agent.complex_agent import ComplexAgent
out = ComplexAgent().solve('少陰病寒化與熱化怎麼區分？分別有哪些主方、誤治風險和條文依據？',
                           role='researcher')
print('規劃器:', out['plan']['planner'])
for t in out['subtasks']:
    print(f"  {t['id']} [{t['kind']:<13}] 依賴{t['depends_on'] or '—'} 工具{t['tools_used']}")
print('成功標準:', out['plan']['success_criteria'][-2])
print('覆蓋檢查 unmet:', out['criteria_check']['unmet'],
      '| 引用核驗:', out['citation_report']['ok'],
      '| 證據', len(out['evidence_clause_ids']), '條')


In [ ]:
# 6.3 會話記憶：跨輪指代消解 +「不是X而是Y」糾錯記憶
from hermes_shanghan.agent.session import AgentSession
sess = AgentSession(role='researcher')
sess.ask('桂枝湯的方證要點是什麼？')
out = sess.ask('它的劑量比呢？')
print('指代消解:', out['session']['contextualized'], '| 錨點:', out['session']['anchors'])
sess.ask('不是桂枝加芍藥湯，而是桂枝去芍藥湯，它的加減關係呢？')
print('糾錯記憶:', sess.corrections)
print(out['answer'][:300])


In [ ]:
# 6.4 深度研究循環：問題細化 → 子代理取證 → 批評家查六維度缺口 → 收斂 + 缺口建議
from hermes_shanghan.agent.research_loop import DeepResearcher
dossier = DeepResearcher().run('桂枝湯類方的劑量演化與注家詮釋')
print('研究問題細化:')
for q in dossier['research_questions'][:3]:
    print('  ·', q)
print(f"{dossier['n_rounds']} 輪收斂 | 覆蓋: {dossier['coverage']} | 證據 {len(dossier['evidence_clause_ids'])} 條")
for f in dossier['findings']:
    print(f" [{f['dimension']}] {f['summary'][:64]}")
for g in dossier['gap_report']:
    print(' 缺口:', g['dimension'], '→', g['suggestion'])


In [ ]:
# 6.5 多假設方證鑒別：並列假設 + 反證 + 缺失核心證 + 鑒別追問（不再單一答案）
from hermes_shanghan.agent.tools import get_registry
hyp = get_registry().call('shanghan_hypotheses',
                          {'symptoms': ['惡寒', '發熱', '無汗', '身疼痛'],
                           'pulse': ['脈浮緊']})
for h in hyp['hypotheses']:
    print(f"{h['formula']:<10} 置信{h['confidence']} 分{h['score']:<5} "
          f"反證示例: {h['counter_evidence_would_be'][:1]}")
print('決策:', hyp['decision'], '| 需要追問:', hyp['needs_clarification'])
for q in hyp['clarifying_questions'][:4]:
    print('  追問:', q)


In [ ]:
# 6.6 多智能體合議：專家獨立判斷 → ConsensusJudge 共識/分歧/需補充 + 評分裁決
from hermes_shanghan.agent.multi_agent import Council
out = Council().deliberate('病人惡寒發熱無汗身疼痛，脈浮緊，會不會誤下成壞病？',
                           role='doctor')
adj = out['consensus']
print('主導假設:', adj['dominant_hypothesis'],
      '| 置信:', adj['final_confidence'], '|', adj['decision'])
print('專家判斷:', [(j['agent'], j['hypothesis'], j['confidence']) for j in out['judgments']])
print('分歧:', adj['disagreements'])
print('需補充:', adj['must_verify'][:3])


## 7 · 一鍵學術溯源論文 + SVG 統計圖表


In [ ]:
from hermes_shanghan.orchestrator import Artifacts
from hermes_shanghan.paper.writer import PaperWriter
art = Artifacts()
w = PaperWriter(art.clauses, art.initial_rules, art.formula_rules,
                art.six_channel_rules, art.mistreatment_rules,
                art.differential_rules, commentary_rules=art.commentary_rules)
path = w.generate(paper_type='provenance', topic='桂枝湯類方源流')
print('manuscript:', path)


In [ ]:
# 統計圖表內聯展示（高頻方 / 一致度熱圖 / 劑量三家折算 / 評測基準）
from IPython.display import SVG, display
for fig in ('fig5_formula_frequency', 'fig7_dose_totals', 'fig8_benchmark'):
    display(SVG(filename=str(path.parent / (fig + '.svg'))))


In [ ]:
# 論文正文預覽（前 3500 字）
from IPython.display import Markdown
Markdown(path.read_text(encoding='utf-8')[:3500])


## 8 · Web 控制台（iframe）· 工具規格導出 · MCP


In [ ]:
# 8.1 在 Colab 內啟動 Web 控制台（11 模塊 SPA）
import threading
from hermes_shanghan.server.http_server import serve
threading.Thread(target=serve, kwargs={'host': '127.0.0.1', 'port': 8765,
                                       'warm': False}, daemon=True).start()
import time; time.sleep(2)
try:
    from google.colab import output           # Colab 環境
    output.serve_kernel_port_as_iframe(8765, height=700)
except ImportError:
    print('非 Colab 環境：瀏覽器打開 http://127.0.0.1:8765/')


In [ ]:
# 8.2 導出 OpenAI/Anthropic 工具規格（24 工具）；MCP 一行接入 Claude Code：
#     claude mcp add shanghan -- python3 -m hermes_shanghan serve-mcp
from hermes_shanghan.integrations.tool_specs import openai_tool_specs
print([s['function']['name'] for s in openai_tool_specs()])


## 9 · 🌸 粉晶 Gradio 界面（醫哲未來人工智能研究院）· ngrok 公網映射

**Rose-Quartz Studio** —— 一個界面集成全部能力：

| 分區 | 內容 |
|---|---|
| 💬 對話研習 | 單智能體 / 任務圖編排 / 多智能體合議 × 醫師/科研/學生/患者四角色；右側四聯面板：**檢索原文**（條文卡片可回源）· 多假設 · 合議 · 核驗 · 軌跡 |
| 🔭 深度研究 | 研究問題細化 → 六維度取證 → 缺口報告，檔案一鍵導出 Markdown/JSON |
| ⚗️ 方證工具台 | 原文檢索 · 條文全息（異文/注家/關係）· 多假設匹配 · 鑒別 · 劑量 · 禁忌 |
| 📊 評測基準 | 四套件指標速覽 + 智能體基準一鍵重跑 |

填入 [ngrok authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) 可獲公網固定鏈接；
留空則使用 gradio 官方分享鏈接。對話與研究結果均可**導出**。


In [ ]:
#@title 🌸 一鍵啟動粉晶界面（可選填 ngrok token） { display-mode: "form" }
NGROK_AUTHTOKEN = ''  #@param {type:"string"}
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'gradio>=6.0', 'pyngrok>=7.0'], check=True)
from hermes_shanghan.apps.webui import launch_webui
app, url = launch_webui(ngrok_token=NGROK_AUTHTOKEN.strip() or None,
                        share=not NGROK_AUTHTOKEN.strip(), quiet=False)
print('🌸 界面已啟動：', url or '見上方 gradio 公網鏈接（*.gradio.live）')


## 10 ·（可選）中醫笈成全庫：自動下載 + 快速查閱

800+ 部醫籍的文獻旁證層（非經文層）。一條命令自動下載（69MB，sha256 校驗 →
解壓 → 編目 → 字符倒排索引），之後毫秒級編目檢索、全文檢索（書·章節定位）、
按章閱讀。智能體工具 `shanghan_library` 與 CLI `library` 共用同一能力面。

In [ ]:
#@title 下載並查閱全庫（約 1-2 分鐘；跳過請不運行） { display-mode: "form" }
from hermes_shanghan.corpus import library

library.fetch()                      # 冪等：已就緒則直接返回
lib = library.Library()
print('全庫：', lib.catalog['n_books'], '部 /', lib.catalog['n_units'], '個文本單元')

# 編目檢索：書名/作者/朝代/分類（異體字折疊）
for h in lib.search('金匱', limit=4):
    print(f"  《{h['title']}》{h['author']}·{h['dynasty']} [{h['category']}]")

# 全文檢索：字符索引剪枝 → 逐書驗證 → 書·章節定位摘錄
out = lib.grep('奔豚', category='醫案', limit=3)
for h in out['hits']:
    print(f"  《{h['title']}》§{h['section'][:16]} …{h['excerpt'][:42]}…")

# 按章閱讀 + 智能體自動路由到 shanghan_library
print(lib.read('傷寒來蘇集', section='傷寒總論', max_chars=160)['text'][:160])
from hermes_shanghan.agent.agent import ShanghanAgent
ans = ShanghanAgent().ask('歷代醫書中哪些書記載了奔豚？', role='researcher')
print('tools_used:', ans['tools_used'])


---
**四項保證**：證據回源（每個 clause_id 可點回原文）· A/B/C/D/E 層級標註 ·
患者安全（診斷/處方/劑量上游攔截）· 優雅降級（無 key 自動 local 後端）。

倉庫: https://github.com/pariskang/Shanghan-Hermes ｜ 220 項測試 ｜ 字節級可復現
